# 04 — Accommodation and Business Capacity
## Trade Winds & Headwinds: USVI, 2000–2025

Sections:
1. USVIBER room inventory and hotel count (survey-based, annual)
2. Census guestroom crosswalk (NAICS 7211, 2007–2022)
3. Census establishment counts and receipts
4. Occupancy and utilization
5. Room inventory vs. visitor volume

**Source divergence note:** USVIBER counts all accommodation providers via annual survey. Census NAICS 7211 covers only payroll-filing establishments. Counts differ (~30% gap at comparable years). Both series are retained and labeled separately.

In [1]:
import sys, importlib
from pathlib import Path
import os

PROJ_ROOT = Path("..").resolve()
if str(PROJ_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJ_ROOT))
os.chdir(PROJ_ROOT)

import numpy as np
import pandas as pd
import plotly.graph_objects as go

import src.cleaning as _c
import src.charts as _ch
import src.hotel_extraction as _he

importlib.reload(_c)
importlib.reload(_ch)
importlib.reload(_he)

from src.cleaning import (
    load_usviber_combined,
    load_census_rooms,
    load_census_2007,
    load_census_2012_basic,
    load_census_2017_basic,
    load_census_2022_basic,
)
from src.hotel_extraction import load_hotel_annual, build_hotel_annual
from src.charts import PALETTE, bar_chart, line_chart, add_shocks, source_note

# build_hotel_annual()
# print("Hotel annual data built.")

pd.set_option("display.float_format", "{:,.1f}".format)
print("Ready.")


Ready.


In [2]:
tour = load_usviber_combined()
rooms = load_census_rooms()
c07 = load_census_2007()
c12 = load_census_2012_basic()
c17 = load_census_2017_basic()
c22 = load_census_2022_basic()

# Deduplicated Census NAICS 72/721/722
frames = []
for df, v in [(c07, 2007), (c12, 2012), (c17, 2017), (c22, 2022)]:
    sub = df[df["naics"].astype(str).isin(["72", "721", "722"])][
        ["year", "naics", "naics_label", "estab", "rcptot_k", "payann_k", "emp"]
    ].copy()
    sub["vintage"] = v
    frames.append(sub)
census = (
    pd.concat(frames)
    .drop_duplicates(subset=["year", "naics"])
    .sort_values(["naics", "year"])
)

print("Census NAICS 721 (Accommodation):")
print(
    census[census["naics"] == "721"][
        ["year", "estab", "emp", "payann_k", "rcptot_k"]
    ].to_string(index=False)
)

Census NAICS 721 (Accommodation):
 year  estab     emp  payann_k  rcptot_k
 2012     57 3,876.0 111,879.0 325,265.0
 2017     51 3,385.0  99,222.0 320,666.0
 2022     57 1,650.0  55,544.0 240,810.0


---
## 1. USVIBER Room Inventory

In [3]:
fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=tour["year"],
        y=tour["hotel_rooms"],
        name="Hotel rooms",
        marker_color=PALETTE["rooms"],
    )
)
fig.add_trace(
    go.Bar(
        x=tour["year"],
        y=tour["condo_units"],
        name="Condo & other units",
        marker_color=PALETTE["cruise"],
    )
)
fig.update_layout(
    barmode="stack",
    title=dict(
        text="USVI Accommodation Inventory — USVIBER Survey<br>"
        '<span style="font-size:13px;color:#555">Annual survey. Hotels + condo/vacation rental units.</span>',
        font=dict(size=17),
    ),
    yaxis_title="Rooms or units",
    yaxis_tickformat=",",
    height=440,
    bargap=0.2,
    annotations=[
        dict(
            xref="paper",
            yref="paper",
            x=0,
            y=-0.08,
            text='<span style="color:#888;font-size:11px">Source: USVIBER Annual Tourism Indicator 2024</span>',
            showarrow=False,
            xanchor="left",
        )
    ],
)
for x_val, label in [(2017, "Irma & Maria"), (2020, "COVID-19")]:
    fig.add_vline(
        x=x_val,
        line=dict(color="#999", width=1.5, dash="dash"),
        annotation_text=label,
        annotation_position="top",
        annotation_font=dict(size=10, color="#666"),
    )
fig.show()

In [4]:
fig = bar_chart(
    tour,
    x="year",
    y="hotel_count",
    title="Number of Hotels — USVIBER Survey",
    y_label="Number of hotels",
    color=PALETTE["rooms"],
    source="USVIBER Annual Tourism Indicator 2024",
    shock_x=[2017, 2020],
)
fig.show()
print(
    tour[["year", "hotel_count", "hotel_rooms", "condo_units", "total_rooms"]]
    .set_index("year")
    .to_string()
)

      hotel_count  hotel_rooms  condo_units  total_rooms
year                                                    
2000           49         3947         1061         5008
2001           48         3987         1066         5053
2002           47         4036         1102         5138
2003           46         3985         1059         5045
2004           48         3906         1053         4959
2005           47         3687         1075         4761
2006           47         3739         1073         4812
2007           46         3678         1079         4757
2008           47         3782         1075         4857
2009           46         3809         1139         4948
2010           47         3878         1131         5008
2011           47         3759         1130         4889
2012           46         3874         1157         5030
2013           44         3897         1120         5016
2014           44         3856         1118         4974
2015           41         3903 

---
## 2. Census Guestroom Crosswalk

| Vintage | Method | Rooms | Establishments |
|---|---|---|---|
| 2007 | A10 industry total (PSCODE 20000) | 4,065 | 47 |
| 2012 | A10 industry total | 5,027 | 54 |
| 2017 | NAPCS 7003825000 (room accommodation) — **Dec 31, 2017, post-hurricane** | 2,624 | 46 |
| 2022 | NAPCS 7003825000 | 2,039 | 51 |

Cross-vintage comparability is approximate; table structures differ between 2007/2012 and 2017/2022.

In [5]:
# Overlay Census rooms with USVIBER hotel rooms
fig = go.Figure()

# USVIBER hotel rooms (annual)
fig.add_trace(
    go.Scatter(
        x=tour["year"],
        y=tour["hotel_rooms"],
        mode="lines",
        name="USVIBER hotel rooms (annual survey)",
        line=dict(color=PALETTE["rooms"], width=2),
    )
)
fig.add_trace(
    go.Scatter(
        x=tour["year"],
        y=tour["total_rooms"],
        mode="lines",
        name="USVIBER total rooms (hotels + condos)",
        line=dict(color=PALETTE["total"], width=2, dash="dash"),
    )
)
# Census rooms (quinquennial snapshots)
fig.add_trace(
    go.Scatter(
        x=rooms["year"],
        y=rooms["rooms"],
        mode="markers",
        name="Census NAICS 7211 rooms (Dec 31)",
        marker=dict(color=PALETTE["cruise"], size=12, symbol="diamond"),
    )
)

fig.update_layout(
    title=dict(
        text="Room Inventory: USVIBER Survey vs. Census NAICS 7211<br>"
        '<span style="font-size:13px;color:#555">Two different measurements of accommodation capacity  different scope and methodology</span>',
        font=dict(size=17),
    ),
    yaxis_title="Rooms / units",
    yaxis_tickformat=",",
    height=460,
    annotations=[
        dict(
            xref="paper",
            yref="paper",
            x=0,
            y=-0.08,
            text='<span style="color:#888;font-size:11px">Sources: USVIBER Annual Tourism Indicator 2024; U.S. Census Economic Census 2007-2022</span>',
            showarrow=False,
            xanchor="left",
        )
    ],
)
for x_val, label in [(2017, "Irma & Maria"), (2020, "COVID-19")]:
    fig.add_vline(
        x=x_val,
        line=dict(color="#999", width=1.5, dash="dash"),
        annotation_text=label,
        annotation_position="top",
        annotation_font=dict(size=10, color="#666"),
    )
fig.show()

print("\nCensus rooms crosswalk:")
print(
    rooms[["year", "rooms", "estab", "rcptot_k", "methodology_note"]].to_string(
        index=False
    )
)


Census rooms crosswalk:
 year  rooms  estab  rcptot_k                                                                                                                methodology_note
 2007   4065     47    308481                                                                                               A10 industry total (PSCODE 20000)
 2012   5027     54    319049                                                                                         A10 industry total (PSCODE2012 = 20000)
 2017   2624     46    237838 NAPCS02 product 7003825000 (Room or unit accommodation for travelers). Guestroom reference date = Dec 31, 2017, post-hurricane.
 2022   2039     51    135204                                                           NAPCS02 product 7003825000 (Room or unit accommodation for travelers)


---
## 3. Census Establishments and Receipts

In [6]:
# Establishments and receipts for NAICS 72, 721, 722
fig = go.Figure()
for naics_code, name, color in [
    ("72", "NAICS 72 (Accommodation + food)", PALETTE["total"]),
    ("721", "NAICS 721 (Accommodation only)", PALETTE["tourists"]),
    ("722", "NAICS 722 (Food services)", PALETTE["cruise"]),
]:
    sub = (
        census[census["naics"] == naics_code]
        .dropna(subset=["estab"])
        .sort_values("year")
    )
    fig.add_trace(
        go.Scatter(
            x=sub["year"],
            y=sub["estab"],
            mode="markers+lines",
            name=name,
            line=dict(color=color, width=2, dash="dot"),
            marker=dict(size=10, symbol="diamond"),
        )
    )
fig.update_layout(
    title=dict(
        text="Census Establishment Counts<br>"
        '<span style="font-size:13px;color:#555">Quinquennial. Only payroll-filing establishments.</span>',
        font=dict(size=17),
    ),
    yaxis_title="Number of establishments",
    height=420,
    annotations=[
        dict(
            xref="paper",
            yref="paper",
            x=0,
            y=-0.08,
            text='<span style="color:#888;font-size:11px">Source: U.S. Census Economic Census 2007-2022 — USVI territory level</span>',
            showarrow=False,
            xanchor="left",
        )
    ],
)
for x_val, label in [(2017, "Irma & Maria"), (2020, "COVID-19")]:
    fig.add_vline(
        x=x_val,
        line=dict(color="#999", width=1.5, dash="dash"),
        annotation_text=label,
        annotation_position="top",
        annotation_font=dict(size=10, color="#666"),
    )
fig.show()

In [7]:
# Accommodation vs food services receipts split
c_721 = census[census["naics"] == "721"][["year", "rcptot_k"]].rename(
    columns={"rcptot_k": "accom_k"}
)
c_722 = census[census["naics"] == "722"][["year", "rcptot_k"]].rename(
    columns={"rcptot_k": "food_k"}
)
rcpt = c_721.merge(c_722, on="year").dropna()
rcpt["accom_m"] = rcpt["accom_k"] / 1000
rcpt["food_m"] = rcpt["food_k"] / 1000

fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=rcpt["year"],
        y=rcpt["accom_m"],
        name="Accommodation (NAICS 721)",
        marker_color=PALETTE["tourists"],
    )
)
fig.add_trace(
    go.Bar(
        x=rcpt["year"],
        y=rcpt["food_m"],
        name="Food services (NAICS 722)",
        marker_color=PALETTE["cruise"],
    )
)
fig.update_layout(
    barmode="stack",
    title=dict(
        text="Census Receipts: Accommodation vs. Food Services (Nominal)<br>"
        '<span style="font-size:13px;color:#555">Millions of nominal USD. Quinquennial snapshots.</span>',
        font=dict(size=17),
    ),
    yaxis_title="Receipts ($M, nominal)",
    height=420,
    annotations=[
        dict(
            xref="paper",
            yref="paper",
            x=0,
            y=-0.08,
            text='<span style="color:#888;font-size:11px">Source: U.S. Census Economic Census 2007-2022 — USVI</span>',
            showarrow=False,
            xanchor="left",
        )
    ],
)
for x_val, label in [(2017, "Irma & Maria")]:
    fig.add_vline(
        x=x_val,
        line=dict(color="#999", width=1.5, dash="dash"),
        annotation_text=label,
        annotation_position="top",
        annotation_font=dict(size=10, color="#666"),
    )
fig.show()

---
## 4. Occupancy and Utilization

In [8]:
# Occupancy rate alongside available rooms
fig = go.Figure()
fig.add_trace(
    go.Bar(
        x=tour["year"],
        y=tour["total_rooms"],
        name="Available rooms/units",
        marker_color="#c8e6f7",
        yaxis="y",
    )
)
fig.add_trace(
    go.Scatter(
        x=tour["year"],
        y=tour["occupancy_rate_pct"],
        mode="lines+markers",
        name="Occupancy rate (%)",
        line=dict(color=PALETTE["occupancy"], width=2.5),
        yaxis="y2",
    )
)
fig.update_layout(
    title=dict(
        text="Room Inventory and Occupancy Rate<br>"
        '<span style="font-size:13px;color:#555">Bars = available rooms (left axis); line = occupancy % (right axis)</span>',
        font=dict(size=17),
    ),
    yaxis=dict(title="Available rooms/units", tickformat=","),
    yaxis2=dict(
        title="Occupancy rate (%)",
        overlaying="y",
        side="right",
        range=[0, 100],
        showgrid=False,
    ),
    height=460,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    annotations=[
        dict(
            xref="paper",
            yref="paper",
            x=0,
            y=-0.08,
            text='<span style="color:#888;font-size:11px">Source: USVIBER Annual Tourism Indicator 2024</span>',
            showarrow=False,
            xanchor="left",
        )
    ],
)
for x_val, label in [(2017, "Irma & Maria"), (2020, "COVID-19")]:
    fig.add_vline(
        x=x_val,
        line=dict(color="#999", width=1.5, dash="dash"),
        annotation_text=label,
        annotation_position="top",
        annotation_font=dict(size=10, color="#666"),
    )
fig.show()

---
## 5. Room Inventory vs. Visitor Volume

In [9]:
# Tourists per available room — rough utilization indicator
# (tourists in thousands, rooms in units — multiply tourists by 1000 first)
tour2 = tour.copy()
tour2["tourists_per_room"] = (tour2["tourists"] * 1000) / tour2["total_rooms"]

fig = bar_chart(
    tour2,
    x="year",
    y="tourists_per_room",
    title="Annual Overnight Tourists per Available Room",
    subtitle="Rough utilization indicator. Assumes constant room availability across the year.",
    y_label="Annual tourists per room",
    color=PALETTE["tourists"],
    source="USVIBER Annual Tourism Indicator 2024 — author calculations",
    shock_x=[2017, 2020],
)
fig.show()
print(
    tour2[["year", "tourists", "total_rooms", "tourists_per_room"]]
    .set_index("year")
    .round(1)
    .to_string()
)

      tourists  total_rooms  tourists_per_room
year                                          
2000     546.7         5008              109.2
2001     527.2         5053              104.3
2002     520.2         5138              101.2
2003     537.9         5045              106.6
2004     566.8         4959              114.3
2005     593.8         4761              124.7
2006     570.6         4812              118.6
2007     587.1         4757              123.4
2008     574.2         4857              118.2
2009     562.6         4948              113.7
2010     576.2         5008              115.1
2011     566.9         4889              116.0
2012     619.2         5030              123.1
2013     590.3         5016              117.7
2014     614.8         4974              123.6
2015     641.9         5060              126.9
2016     667.9         4842              137.9
2017     535.0         4886              109.5
2018     380.9         3255              117.0
2019     418.

---
## 6. USVIBER Hotel Utilization (PDF Series)

Monthly hotel statistics extracted from USVIBER Hotel Indicator PDFs:
- **Room nights available** (`RA` series): 2006–2025.
- **Room nights occupied** (`RN` / `HRN` series): 2000–2025 (gap: year 02, partial 2025).
- **Occupancy rates** (`H` series): 2000–2025 (gaps: year 03, year 18 missing PDFs, partial 2025).


In [10]:
# Load hotel data — rebuild from PDFs if interim files are missing
hotel_usvi   = load_hotel_annual()

print("=== Hotel series coverage ===")
print(f"USVI Total years: {hotel_usvi.year.min()}–{hotel_usvi.year.max()}")
print()
print("USVI Total (2014–2024), showing reporting_period for occupancy:")
print(hotel_usvi[hotel_usvi.year.between(2014, 2024)].to_string(index=False))

print()
print("=== Occupancy comparison (PDF vs room nights vs Annual Indicator) ===")
tour = load_usviber_combined()
comp = hotel_usvi[["year","occupancy_pct","occ_calculated"]].merge(
    tour[["year","occupancy_rate_pct"]], on="year", how="left"
)
comp["diff_ppt"] = (comp["occupancy_pct"] - comp["occupancy_rate_pct"]).round(1)
comp = comp.rename(columns={
    "occupancy_pct": "PDF_annual_total",
    "occ_calculated": "occ_nights_based",
    "occupancy_rate_pct": "usviber_annual_indicator",
})
print(comp[comp.year >= 2016].to_string(index=False))
print()
print("The PDF-reported occupancy, room-nights calculation, and Annual Tourism")
print("Indicator are presented as separate measures. Their coverage and")
print("methodological comparability have not been established.")


=== Hotel series coverage ===
USVI Total years: 2001–2025

USVI Total (2014–2024), showing reporting_period for occupancy:
 year available_filename  available_prev_year_value  nights_available_k  occupied_filename  occupied_prev_year_value  nights_occupied_k occupancy_filename  occupancy_prev_year_value  occupancy_pct  occ_calculated
 2014       RA14-revised                    1,792.4             1,754.2       RN14-REVISED                     966.1              946.0                H14                       53.9           53.2            53.9
 2015       RA15-revised                    1,754.2             1,733.7       RN15-REVISED                     946.0              989.9        H15-REVISED                       53.9           57.1            57.1
 2016       RA16-revised                    1,733.7             1,742.8       RN16-REVISED                     989.9              988.8        H16-REVISED                       57.1           56.7            56.7
 2017       RA17-revised 

In [11]:

# Prepare the three measures on a common year index.
tour_occupancy = (
    tour[["year", "occupancy_rate_pct"]]
    .copy()
    .assign(year=lambda df: pd.to_numeric(df["year"], errors="coerce"))
    .dropna(subset=["year"])
)

tour_occupancy["year"] = tour_occupancy["year"].astype(int)

if tour_occupancy["year"].duplicated().any():
    raise ValueError("Annual Tourism Indicator data contains duplicate years.")

occupancy_comparison = (
    hotel_usvi[["year", "occupancy_pct", "occ_calculated"]]
    .merge(tour_occupancy, on="year", how="outer", validate="one_to_one")
    .query("2000 <= year <= 2025")
    .sort_values("year")
    .reset_index(drop=True)
)

# Plot
fig = go.Figure()

series = [
    ("PDF-reported occupancy", "occupancy_pct"),
    ("Calculated from room nights", "occ_calculated"),
    ("USVIBER Annual Tourism Indicator", "occupancy_rate_pct"),
]

for label, column in series:
    fig.add_trace(
        go.Scatter(
            x=occupancy_comparison["year"],
            y=occupancy_comparison[column],
            mode="lines+markers",
            name=label,
            connectgaps=False,
        )
    )

fig.update_layout(
    title="USVI Hotel Occupancy: Three Measures, 2000–2025",
    xaxis_title="Year",
    yaxis_title="Occupancy (%)",
    yaxis=dict(range=[0, 100], ticksuffix="%"),
    xaxis=dict(
        tickmode="linear",
        dtick=1,
    ),
    hovermode="x unified",
    legend_title_text="Measure",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.3,
        xanchor="center",
        x=0.5
    ),
    template="plotly_white",
)

fig.show()

# Comparison table
comparison_table = occupancy_comparison.copy()

comparison_table["Calculated − PDF (pp)"] = (
    comparison_table["occ_calculated"] - comparison_table["occupancy_pct"]
).round(1)

comparison_table["Indicator − PDF (pp)"] = (
    comparison_table["occupancy_rate_pct"] - comparison_table["occupancy_pct"]
).round(1)

comparison_table = comparison_table.rename(
    columns={
        "year": "Year",
        "occupancy_pct": "PDF-reported (%)",
        "occ_calculated": "Room-nights calculated (%)",
        "occupancy_rate_pct": "Annual Indicator (%)",
    }
)[
    [
        "Year",
        "PDF-reported (%)",
        "Room-nights calculated (%)",
        "Annual Indicator (%)",
        "Calculated − PDF (pp)",
        "Indicator − PDF (pp)",
    ]
]

display(
    comparison_table.style.format(
        {
            "PDF-reported (%)": "{:.1f}",
            "Room-nights calculated (%)": "{:.1f}",
            "Annual Indicator (%)": "{:.1f}",
            "Calculated − PDF (pp)": "{:+.1f}",
            "Indicator − PDF (pp)": "{:+.1f}",
        },
        na_rep="—",
    )
)

,Year,PDF-reported (%),Room-nights calculated (%),Annual Indicator (%),Calculated − PDF (pp),Indicator − PDF (pp)
0,2000,—,—,57.7,—,—
1,2001,60.8,—,56.2,—,-4.6
2,2002,56.9,—,56.6,—,-0.3
3,2003,—,—,57.2,—,—
4,2004,61.8,—,61.3,—,-0.5
5,2005,64.3,—,63.8,—,-0.5
6,2006,60.6,—,60.1,—,-0.5
7,2007,64.9,64.9,64.4,+0.0,-0.5
8,2008,60.0,60.0,60.0,+0.0,+0.0
9,2009,55.0,55.0,54.8,+0.0,-0.2


In [12]:
# --- Nights available and occupied (USVI total) ---
fig = go.Figure()
fig.add_trace(go.Bar(
    x=hotel_usvi["year"], y=hotel_usvi["nights_available_k"],
    name="Room nights available", marker_color="#c8e6f7", opacity=0.85,
))
fig.add_trace(go.Scatter(
    x=hotel_usvi["year"], y=hotel_usvi["nights_occupied_k"],
    mode="lines", name="Room nights occupied",
    line=dict(color=PALETTE["tourists"], width=2.4),
))
fig.update_layout(
    title=dict(text="<b>Hotel Room Nights Available and Occupied (USVI Total)</b>", font=dict(size=16), x=0.0),
    yaxis_title="Thousands of room nights",
    yaxis_tickformat=",",
    height=420,
    barmode="overlay",
    annotations=[source_note("USVIBER Hotel Indicator PDFs (RA and RN series). 2025 excluded (partial year).")],
)
add_shocks(fig)
fig.show()


In [13]:
# --- Occupancy: Annual_Total (corrected) vs calculated vs USVIBER annual ---
# After correcting extraction to use ANNUAL_TOTAL rows:
#   - 2017: PDF Annual_Total = 44.1% (matches USVIBER 43.9%; old code had 51.3%)
#   - 2018-2019: PDF still higher than USVIBER due to population coverage (reporting hotels
#     running at high utilization; USVIBER includes broader market with damaged capacity)
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=hotel_usvi["year"], y=hotel_usvi["occ_calculated"],
    mode="lines+markers", name="Nights-based (occ/avail)",
    line=dict(color=PALETTE["air_visitors"], width=2.4),
    marker=dict(size=5),
))
fig.add_trace(go.Scatter(
    x=hotel_usvi["year"], y=hotel_usvi["occupancy_pct"],
    mode="lines", name="PDF Annual_Total",
    line=dict(color=PALETTE["total"], width=1.6, dash="dash"),
    opacity=0.75,
))
fig.add_trace(go.Scatter(
    x=tour["year"], y=tour["occupancy_rate_pct"],
    mode="lines+markers", name="USVIBER Annual Indicator",
    line=dict(color=PALETTE["cruise"], width=2.0, dash="dot"),
    marker=dict(size=4),
))
fig.update_layout(
    title=dict(
        text="<b>Hotel Occupancy Rate — Three Measures</b><br>"
             "<span style=\'font-size:12px;color:#6b7280\'>PDF Annual_Total and nights-based rates cover reporting hotels only."
             " USVIBER Annual Indicator covers the full accommodation market.</span>",
        font=dict(size=16), x=0.0,
    ),
    yaxis_title="Occupancy rate (%)",
    yaxis_range=[0, 90],
    height=440,
    annotations=[source_note(
        "USVIBER Hotel PDFs (Annual_Total row); USVIBER Annual Tourism Indicator 2024."
    )],
)
add_shocks(fig)
fig.show()


In [14]:
# --- Capacity utilization index (2016 = 100) ---
BASE = 2016

def _idx(series, base=BASE):
    bv = series.loc[series.index == base]
    if bv.empty or bv.values[0] == 0:
        return series * float("nan")
    return series / bv.values[0] * 100

h = hotel_usvi.set_index("year")

fig = go.Figure()
for col, label, color, dash in [
    ("nights_available_k", "Room nights available", PALETTE["rooms"],       "solid"),
    ("nights_occupied_k",  "Room nights occupied",  PALETTE["employment"],  "solid"),
    ("occ_calculated",     "Occupancy rate",        PALETTE["air_visitors"],"solid"),
]:
    idx_s = _idx(h[col].dropna())
    fig.add_trace(go.Scatter(
        x=idx_s.index, y=idx_s.values,
        mode="lines", name=label,
        line=dict(color=color, width=2.0, dash=dash),
    ))

fig.add_hline(y=100, line=dict(color="#d1d5db", width=1, dash="dot"),
              annotation_text="2016 = 100", annotation_position="right",
              annotation_font=dict(size=9, color="#aaa"))
fig.update_layout(
    title=dict(text="<b>Hotel Capacity Utilization — Recovery Index (2016 = 100)</b>", font=dict(size=16), x=0.0),
    yaxis_title="Index (2016 = 100)",
    height=420,
    annotations=[source_note(
        "USVIBER Hotel Indicator PDFs. Occupancy index = proportional change in occupancy rate, not percentage points."
    )],
)
add_shocks(fig)
fig.show()
